# Empirical CAPM Testing on the Russian Stock Market (2018–2024)

## Методология
1. **Time-Series OLS** (Sharpe, 1964; Lintner, 1965): для каждой акции оцениваем α и β
2. **Fama-MacBeth Two-Pass** (1973): проверяем, оценивает ли рынок бета-риск
3. **GRS Test** (Gibbons, Ross, Shanken, 1989): совместная значимость всех альф

**CAPM предсказывает:**
- $\alpha_i = 0$ для всех активов
- $\gamma_0 = 0$ (нулевая бета = безрисковая ставка)
- $\gamma_1 = E[R_m - r_f]$ (бета полностью объясняет кросс-секцию)

**Данные:** Yahoo Finance (.ME), ОФЗ 1Y как $r_f$, рыночный индекс IMOEX.ME

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. ИМПОРТЫ И НАСТРОЙКИ
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

print('Все библиотеки загружены.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. КОНФИГУРАЦИЯ
# ─────────────────────────────────────────────────────────────────────────────
START = '2018-01-01'
END   = '2024-12-31'

# Топ-30 ликвидных бумаг ММВБ (суффикс .ME = Московская биржа на Yahoo Finance)
TICKERS = [
    'SBER.ME',   # Сбербанк
    'GAZP.ME',   # Газпром
    'LKOH.ME',   # Лукойл
    'NVTK.ME',   # Новатэк
    'ROSN.ME',   # Роснефть
    'GMKN.ME',   # Норникель
    'TATN.ME',   # Татнефть
    'CHMF.ME',   # Северсталь
    'NLMK.ME',   # НЛМК
    'ALRS.ME',   # Алроса
    'MTSS.ME',   # МТС
    'VTBR.ME',   # ВТБ
    'PIKK.ME',   # ПИК
    'AFLT.ME',   # Аэрофлот
    'RUAL.ME',   # Русал
    'MOEX.ME',   # Московская биржа
    'PLZL.ME',   # Полюс Золото
    'HYDR.ME',   # РусГидро
    'IRAO.ME',   # Интер РАО
    'MAGN.ME',   # ММК
    'SNGS.ME',   # Сургутнефтегаз
    'PHOR.ME',   # ФосАгро
    'MGNT.ME',   # Магнит
    'SBERP.ME',  # Сбербанк преф.
    'TRNFP.ME',  # Транснефть преф.
    'FEES.ME',   # ФСК ЕЭС
    'YNDX.ME',   # Яндекс (торговался до реструктуризации)
    'TCSG.ME',   # TCS Group (Тинькофф)
    'POSI.ME',   # Positive Technologies
    'OZON.ME',   # Ozon
]

MARKET_TICKER = 'IMOEX.ME'   # Индекс Московской биржи
MIN_OBS_FRAC  = 0.60         # Минимум 60% наблюдений для включения акции

print(f'Акций в выборке: {len(TICKERS)}')
print(f'Период: {START} — {END}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. БЕЗРИСКОВАЯ СТАВКА — ОФЗ 1Y (средняя доходность за год)
# ─────────────────────────────────────────────────────────────────────────────
# Источник: данные ЦБ РФ / Московской биржи (средняя доходность ОФЗ ~1 год)
# https://www.cbr.ru/hd_base/zcyc_params/zcyc/
OFZ_ANNUAL = {
    2018: 0.0730,   # средняя ~7.30%
    2019: 0.0680,   # снижение ключевой ставки
    2020: 0.0520,   # COVID: снижение до 4.25%
    2021: 0.0625,   # начало цикла повышения
    2022: 0.1050,   # санкции: скачок до 20%, затем снижение
    2023: 0.0975,   # ~10% среднегодовая
    2024: 0.1520,   # повышение до 21%
}

# Непрерывно начисляемая месячная ставка: r_monthly = ln(1 + r_annual) / 12
def annual_to_monthly_log(r_annual: float) -> float:
    return np.log(1 + r_annual) / 12

# Строим месячный ряд (начало каждого месяца)
date_range = pd.date_range(start=START, end=END, freq='MS')
rf_series  = pd.Series(
    {d: annual_to_monthly_log(OFZ_ANNUAL[d.year]) for d in date_range},
    name='rf'
)
rf_series.index = pd.PeriodIndex(rf_series.index, freq='M')

print('Безрисковая ставка ОФЗ (месячная, логарифмическая):')
rf_annual_check = rf_series.groupby(rf_series.index.year).mean() * 12 * 100
rf_annual_check.name = 'rf % (annualized)'
print(rf_annual_check.apply(lambda x: f'{x:.2f}%').to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. ЗАГРУЗКА ДАННЫХ С YAHOO FINANCE
# ─────────────────────────────────────────────────────────────────────────────
all_tickers = TICKERS + [MARKET_TICKER]

print('Загружаем котировки (monthly, adjusted)...')
raw = yf.download(
    all_tickers,
    start=START,
    end=END,
    interval='1mo',
    auto_adjust=True,
    progress=True,
    group_by='ticker',
)

# Извлекаем цены закрытия
if isinstance(raw.columns, pd.MultiIndex):
    close = raw.xs('Close', axis=1, level=1)
else:
    close = raw[['Close']]

# Приводим индекс к PeriodIndex(M) для совместимости с rf
close.index = pd.PeriodIndex(close.index, freq='M')

print(f'\nЗагружено: {close.shape[0]} месяцев × {close.shape[1]} инструментов')
print(f'Период: {close.index[0]} — {close.index[-1]}')

# Доля пропущенных значений
miss = close.isnull().mean().sort_values(ascending=False)
print('\nДоля пропусков по инструментам:')
print(miss[miss > 0].apply(lambda x: f'{x*100:.1f}%').to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. ОЧИСТКА, ДОХОДНОСТИ, ФИЛЬТРАЦИЯ
# ─────────────────────────────────────────────────────────────────────────────

# Логарифмические месячные доходности (удобны аддитивностью)
log_ret = np.log(close / close.shift(1)).dropna(how='all')

# Рыночный индекс
market_ret = log_ret[MARKET_TICKER].rename('Rm')
stock_ret  = log_ret[TICKERS]

# Фильтр: не менее MIN_OBS_FRAC от общего числа наблюдений рынка
min_obs      = int(MIN_OBS_FRAC * market_ret.count())
valid_mask   = stock_ret.count() >= min_obs
valid_tickers = stock_ret.columns[valid_mask].tolist()
stock_ret    = stock_ret[valid_tickers]

print(f'Акций прошло фильтр (≥{MIN_OBS_FRAC*100:.0f}% наблюдений): {len(valid_tickers)}')
print(valid_tickers)

# Общий временной индекс
common_idx = (
    market_ret.dropna().index
    .intersection(rf_series.index)
    .intersection(stock_ret.dropna(how='all').index)
)

Rm  = market_ret.loc[common_idx]
rf  = rf_series.loc[common_idx]
R   = stock_ret.loc[common_idx]

# Избыточные доходности
MKT = (Rm - rf).rename('MKT')    # рыночная премия
ER  = R.subtract(rf, axis=0)     # excess returns акций

# Убираем суффикс .ME для читаемости таблиц
clean_name = {t: t.replace('.ME', '') for t in valid_tickers}
ER.rename(columns=clean_name, inplace=True)
R.rename(columns=clean_name,  inplace=True)
names = list(clean_name.values())

print(f'\nИтоговый датасет: {len(common_idx)} месяцев × {len(names)} акций')
print(f'Период: {common_idx[0]} — {common_idx[-1]}')
print(f'\nСредняя рыночная премия (мес.): {MKT.mean()*100:.3f}%')
print(f'Средняя рыночная премия (год):  {MKT.mean()*1200:.2f}%')